In [ ]:
import os
import pandas as pd

BASE_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

print("Files and folders:")
for item in os.listdir(BASE_PATH):
    print(item)

In [ ]:
train = pd.read_csv(f"{BASE_PATH}/train.csv")
train_series = pd.read_csv(f"{BASE_PATH}/train_series.csv")

print("train.csv shape:", train.shape)
print("train_series.csv shape:", train_series.shape)

print("\ntrain.csv:")
display(train.head())

print("\ntrain_series.csv:")
display(train_series.head())

In [ ]:
print("Data types:")
display(train.dtypes)

print("\nMissing values:")
display(train.isnull().sum())

print("\nTarget/label columns:")
for col in train.columns:
    print(f"{col}: {train[col].nunique()} unique values")

In [ ]:
train_series = pd.read_csv(f"{BASE_PATH}/train_series.csv")

print("Shape:", train_series.shape)

print("\nColumns:")
print(train_series.columns.tolist())

print("\nFirst 10 rows:")
display(train_series.head(10))

In [ ]:
print("\nTarget value counts:")
for col in train.columns:
    print("\n", col)
    print(train[col].value_counts(dropna=False).head(10))

In [ ]:
target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("Total studies:", len(train))

labeled_mask = train[target_cols].notna().any(axis=1)

print("Rows with at least one label:", labeled_mask.sum())
print("Rows without any labels:", (~labeled_mask).sum())

display(train.loc[labeled_mask].head())

In [ ]:
print("Labels per study:")

label_count = train[target_cols].notna().sum(axis=1)

print(label_count.value_counts().sort_index())

In [ ]:
labeled_train = train[train[target_cols].notna().any(axis=1)].copy()

# Connect them to their MRI series
labeled_series = labeled_train.merge(
    train_series,
    on="StudyInstanceUID",
    how="inner"
)

print("Labeled studies:", labeled_train.shape)
print("MRI series belonging to labeled studies:", labeled_series.shape)

display(labeled_series.head(10))

In [ ]:
print("MRI anatomical planes:")
display(labeled_series["Anatomical_Plane"].value_counts())

print("\nFluid-sensitive series:")
display(labeled_series["Fluid_Sensitive"].value_counts())

print("\nFat-suppressed series:")
display(labeled_series["Fat_Suppression"].value_counts())

In [ ]:
# Pick the first labeled study
study_id = labeled_train["StudyInstanceUID"].iloc[0]

print("Selected StudyInstanceUID:")
print(study_id)

# Find all MRI series belonging to this study
study_series = train_series[
    train_series["StudyInstanceUID"] == study_id
].copy()

print("\nNumber of series:", len(study_series))

display(study_series)

In [ ]:
print("Searching for this study in train_series...")

train_series_path = f"{BASE_PATH}/train_series"

print("Train series path exists:", os.path.exists(train_series_path))

# Show a few entries in the folder
entries = os.listdir(train_series_path)

print("\nNumber of entries:", len(entries))
print("\nFirst 10 entries:")
print(entries[:10])

In [ ]:
study_path = os.path.join(
    BASE_PATH,
    "train_series",
    study_id
)

print("Study path:")
print(study_path)

print("\nExists:", os.path.exists(study_path))

if os.path.exists(study_path):
    print("\nContents:")
    print(os.listdir(study_path)[:20])

In [ ]:
if os.path.exists(study_path):

    for series_id in os.listdir(study_path):
        series_path = os.path.join(study_path, series_id)

        if os.path.isdir(series_path):
            files = os.listdir(series_path)

            print("\nSeries:", series_id)
            print("Number of files:", len(files))
            print("First few files:", files[:5])

In [ ]:
!pip install -q pydicom

In [ ]:
import pydicom
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Take the first available series from our selected study
series_id = study_series["SeriesInstanceUID"].iloc[0]

series_path = os.path.join(
    BASE_PATH,
    "train_series",
    study_id,
    series_id
)

print("Study:", study_id)
print("Series:", series_id)
print("Path:", series_path)

files = sorted([
    f for f in os.listdir(series_path)
    if f.lower().endswith(".dcm")
])

print("Number of DICOM files:", len(files))
print("First 5 files:", files[:5])

In [ ]:
dicom_path = os.path.join(series_path, files[len(files)//2])

ds = pydicom.dcmread(dicom_path)

print("DICOM information:")
print("Rows:", ds.Rows)
print("Columns:", ds.Columns)
print("Instance Number:", getattr(ds, "InstanceNumber", "N/A"))

image = ds.pixel_array

print("\nImage shape:", image.shape)
print("Minimum:", image.min())
print("Maximum:", image.max())

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(image, cmap="gray")
plt.axis("off")
plt.title("Knee MRI Slice")

plt.show()

In [ ]:
slices = []

for file in files:
    path = os.path.join(series_path, file)

    ds = pydicom.dcmread(path)

    if hasattr(ds, "pixel_array"):
        slices.append(ds)

# Sort slices by InstanceNumber
slices = sorted(
    slices,
    key=lambda x: getattr(x, "InstanceNumber", 0)
)

print("Total slices:", len(slices))

In [ ]:
num_to_show = min(9, len(slices))

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, ax in enumerate(axes.flat):
    if i < num_to_show:
        image = slices[i].pixel_array

        ax.imshow(image, cmap="gray")
        ax.set_title(f"Slice {i + 1}")
    
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
display(
    study_series[
        study_series["SeriesInstanceUID"] == series_id
    ]
)

In [ ]:
import cv2

IMG_SIZE = 224

def preprocess_slice(ds, img_size=IMG_SIZE):
    # Read pixel data
    image = ds.pixel_array.astype(np.float32)

    # Normalize intensity to 0–255
    image -= image.min()

    if image.max() > 0:
        image /= image.max()

    image *= 255.0

    # Resize
    image = cv2.resize(
        image,
        (img_size, img_size),
        interpolation=cv2.INTER_AREA
    )

    # Convert to 3 channels for pretrained CNNs
    image = np.stack([image] * 3, axis=-1)

    # Normalize to 0–1
    image = image / 255.0

    return image.astype(np.float32)

In [ ]:
processed_image = preprocess_slice(slices[len(slices)//2])

print("Processed shape:", processed_image.shape)
print("Minimum:", processed_image.min())
print("Maximum:", processed_image.max())

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(processed_image)
plt.axis("off")
plt.title("Preprocessed MRI Slice")

plt.show()

In [ ]:
# Keep only studies with at least one available abnormality label
labeled_train = train[
    train[target_cols].notna().any(axis=1)
].copy()

print("Number of labeled studies:", len(labeled_train))

display(
    labeled_train[
        ["StudyInstanceUID", "Report"] + target_cols
    ].head()
)

In [ ]:
label_summary = labeled_train[target_cols].sum().sort_values(ascending=False)

print("Positive cases for each abnormality:")
display(label_summary)

In [ ]:
plt.figure(figsize=(12, 6))

label_summary.plot(kind="bar")

plt.title("Abnormality Distribution")
plt.ylabel("Number of Positive Studies")
plt.xlabel("Abnormality")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
labeled_series = labeled_train[
    ["StudyInstanceUID"] + target_cols
].merge(
    train_series,
    on="StudyInstanceUID",
    how="inner"
)

print("Labeled studies:", labeled_train["StudyInstanceUID"].nunique())
print("Corresponding MRI series:", len(labeled_series))

display(labeled_series.head())

In [ ]:
series_per_study = (
    labeled_series
    .groupby("StudyInstanceUID")
    .size()
)

print("Series per study:")
display(series_per_study.describe())

In [ ]:
# Only use studies where all 12 target labels are available
fully_labeled = train[
    train[target_cols].notna().all(axis=1)
].copy()

print("Fully labeled studies:", len(fully_labeled))

display(
    fully_labeled[
        ["StudyInstanceUID"] + target_cols
    ].head()
)

In [ ]:
baseline_series = train_series[
    (train_series["Fluid_Sensitive"] == 1) &
    (train_series["Anatomical_Plane"] == "Sagittal")
].copy()

print("Available sagittal fluid-sensitive series:",
      len(baseline_series))

display(baseline_series.head())

In [ ]:
baseline_data = fully_labeled[
    ["StudyInstanceUID"] + target_cols
].merge(
    baseline_series,
    on="StudyInstanceUID",
    how="inner"
)

print("Baseline study-series rows:", len(baseline_data))

print(
    "Unique labeled studies:",
    baseline_data["StudyInstanceUID"].nunique()
)

display(baseline_data.head())

In [ ]:
coverage = (
    baseline_data["StudyInstanceUID"]
    .nunique()
)

total_labeled = fully_labeled["StudyInstanceUID"].nunique()

print("Total fully labeled studies:", total_labeled)
print("Studies with sagittal fluid-sensitive series:", coverage)
print("Coverage:", coverage / total_labeled)

In [ ]:
from sklearn.model_selection import train_test_split

study_ids = baseline_data["StudyInstanceUID"].unique()

train_ids, val_ids = train_test_split(
    study_ids,
    test_size=0.20,
    random_state=42
)

train_data = baseline_data[
    baseline_data["StudyInstanceUID"].isin(train_ids)
].copy()

val_data = baseline_data[
    baseline_data["StudyInstanceUID"].isin(val_ids)
].copy()

print("Training studies:", train_data["StudyInstanceUID"].nunique())
print("Validation studies:", val_data["StudyInstanceUID"].nunique())

In [ ]:
# Keep one baseline series for each study
baseline_data = (
    baseline_data
    .sort_values(["StudyInstanceUID", "SeriesInstanceUID"])
    .drop_duplicates("StudyInstanceUID")
    .reset_index(drop=True)
)

print("Studies:", len(baseline_data))
display(baseline_data.head())

In [ ]:
!pip install -q torch torchvision pydicom

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [ ]:
class KneeMRIDataset(Dataset):

    def __init__(
        self,
        dataframe,
        base_path,
        num_slices=8,
        img_size=224
    ):
        self.df = dataframe.reset_index(drop=True)
        self.base_path = base_path
        self.num_slices = num_slices
        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def load_series(self, study_id, series_id):

        series_path = os.path.join(
            self.base_path,
            "train_series",
            study_id,
            series_id
        )

        files = [
            f for f in os.listdir(series_path)
            if f.lower().endswith(".dcm")
        ]

        slices = []

        for file in files:

            path = os.path.join(series_path, file)

            try:
                ds = pydicom.dcmread(path)

                if hasattr(ds, "pixel_array"):
                    slices.append(ds)

            except Exception:
                continue

        # Sort by slice position/instance
        slices.sort(
            key=lambda x: getattr(x, "InstanceNumber", 0)
        )

        return slices

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        study_id = row["StudyInstanceUID"]
        series_id = row["SeriesInstanceUID"]

        slices = self.load_series(
            study_id,
            series_id
        )

        # Select evenly spaced slices
        if len(slices) >= self.num_slices:

            indices = np.linspace(
                0,
                len(slices) - 1,
                self.num_slices
            ).astype(int)

            selected = [slices[i] for i in indices]

        else:

            selected = slices

            # Repeat last slice if necessary
            while len(selected) < self.num_slices:
                selected.append(selected[-1])

        images = []

        for ds in selected:

            image = ds.pixel_array.astype(np.float32)

            image -= image.min()

            if image.max() > 0:
                image /= image.max()

            image = (image * 255).astype(np.uint8)

            image = self.transform(image)

            # Convert grayscale → 3 channels
            image = image.repeat(3, 1, 1)

            images.append(image)

        images = torch.stack(images)

        labels = torch.tensor(
            row[target_cols].values.astype(np.float32)
        )

        return images, labels

In [ ]:
sample_dataset = KneeMRIDataset(
    baseline_data,
    BASE_PATH,
    num_slices=8
)

images, labels = sample_dataset[0]

print("Image tensor shape:", images.shape)
print("Label shape:", labels.shape)

print("\nLabels:")
print(labels)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, ax in enumerate(axes.flat):

    image = images[i].permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(f"Slice {i + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
class KneeMRIModel(nn.Module):

    def __init__(self, num_classes=12):
        super().__init__()

        self.backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Remove ResNet's original classification layer
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # Study-level classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feature_dim, num_classes)
        )

    def forward(self, x):

        # x shape:
        # batch × slices × channels × height × width

        batch_size, num_slices, C, H, W = x.shape

        # Process every slice through ResNet
        x = x.view(
            batch_size * num_slices,
            C,
            H,
            W
        )

        features = self.backbone(x)

        # Restore study/slice structure
        features = features.view(
            batch_size,
            num_slices,
            -1
        )

        # Average slice features
        features = features.mean(dim=1)

        # Predict 12 abnormalities
        output = self.classifier(features)

        return output

In [ ]:
train_dataset = KneeMRIDataset(
    train_data.drop_duplicates("StudyInstanceUID"),
    BASE_PATH,
    num_slices=8
)

val_dataset = KneeMRIDataset(
    val_data.drop_duplicates("StudyInstanceUID"),
    BASE_PATH,
    num_slices=8
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Training studies:", len(train_dataset))
print("Validation studies:", len(val_dataset))

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = KneeMRIModel(
    num_classes=len(target_cols)
).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=2,
    factor=0.5
)

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=loss.item()
        )

    train_loss = (
        running_loss / len(train_loader)
    )

    # Validation
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    print(
        f"\nEpoch {epoch + 1}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Validation Loss = {val_loss:.4f}"
    )

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

model.eval()

all_labels = []
all_probs = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        all_labels.append(
            labels.numpy()
        )

        all_probs.append(
            probabilities.cpu().numpy()
        )

all_labels = np.concatenate(all_labels)
all_probs = np.concatenate(all_probs)

print("Prediction shape:", all_probs.shape)

In [ ]:
for i, target in enumerate(target_cols):

    try:
        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        print(
            f"{target:20s} AUC: {auc:.4f}"
        )

    except ValueError:

        print(
            f"{target:20s} AUC: unavailable"
        )

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("DataLoaders recreated successfully.")

In [ ]:
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()

all_labels = []
all_probs = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        all_labels.append(labels.numpy())
        all_probs.append(probabilities.cpu().numpy())

all_labels = np.concatenate(all_labels)
all_probs = np.concatenate(all_probs)

print("Actual labels shape:", all_labels.shape)
print("Predictions shape:", all_probs.shape)

In [ ]:
for i, target in enumerate(target_cols):

    try:
        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        print(f"{target:20s} AUC: {auc:.4f}")

    except ValueError:
        print(f"{target:20s} AUC: unavailable")

In [ ]:
# Count positive and negative examples in the training studies
y_train = train_data.drop_duplicates("StudyInstanceUID")[target_cols].values

positive_counts = np.sum(y_train == 1, axis=0)
negative_counts = np.sum(y_train == 0, axis=0)

# Avoid division by zero
positive_counts = np.maximum(positive_counts, 1)

pos_weights = negative_counts / positive_counts

print("Positive counts:")
for target, count in zip(target_cols, positive_counts):
    print(f"{target:20s}: {count}")

print("\nPositive-class weights:")
for target, weight in zip(target_cols, pos_weights):
    print(f"{target:20s}: {weight:.2f}")

In [ ]:
pos_weight_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

print("Weighted BCE loss created.")

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "target_cols": target_cols,
        "img_size": 224,
        "num_slices": 8
    },
    "/kaggle/working/knee_mri_baseline.pth"
)

print("Baseline model saved.")

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

print("Backbone frozen.")
print("Classifier trainable.")

In [ ]:
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=1,
    factor=0.5
)

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Validation Loss = {val_loss:.4f}"
    )

In [ ]:
model.eval()

all_labels = []
all_probs = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        all_labels.append(labels.numpy())
        all_probs.append(probabilities.cpu().numpy())

all_labels = np.concatenate(all_labels)
all_probs = np.concatenate(all_probs)

In [ ]:
auc_results = {}

for i, target in enumerate(target_cols):

    try:
        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        auc_results[target] = auc

        print(f"{target:20s} AUC: {auc:.4f}")

    except ValueError:

        auc_results[target] = np.nan

        print(f"{target:20s} AUC: unavailable")

In [ ]:
plane_counts = (
    labeled_series
    .groupby("StudyInstanceUID")["Anatomical_Plane"]
    .nunique()
)

print("Studies with:")
print("3 different planes:", (plane_counts == 3).sum())
print("2 different planes:", (plane_counts == 2).sum())
print("1 plane:", (plane_counts == 1).sum())

In [ ]:
plane_table = pd.crosstab(
    labeled_series["StudyInstanceUID"],
    labeled_series["Anatomical_Plane"]
)

display(plane_table.head(10))

In [ ]:
multi_plane = labeled_series[
    labeled_series["Fluid_Sensitive"] == 1
].copy()

# Prefer one series per study per anatomical plane
multi_plane = (
    multi_plane
    .sort_values(
        ["StudyInstanceUID", "Anatomical_Plane", "SeriesInstanceUID"]
    )
    .drop_duplicates(
        ["StudyInstanceUID", "Anatomical_Plane"]
    )
)

print("Selected series:", len(multi_plane))

display(
    multi_plane[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    ].head(15)
)

In [ ]:
coverage_planes = (
    multi_plane
    .groupby("StudyInstanceUID")["Anatomical_Plane"]
    .nunique()
)

print("Studies with 3 planes:",
      (coverage_planes == 3).sum())

print("Studies with 2 planes:",
      (coverage_planes == 2).sum())

print("Studies with 1 plane:",
      (coverage_planes == 1).sum())

In [ ]:
three_plane_ids = coverage_planes[
    coverage_planes == 3
].index

multi_plane_data = multi_plane[
    multi_plane["StudyInstanceUID"].isin(three_plane_ids)
].copy()

print(
    "Studies available for multi-plane model:",
    multi_plane_data["StudyInstanceUID"].nunique()
)

display(multi_plane_data.head())

In [ ]:
class MultiPlaneKneeDataset(Dataset):

    def __init__(
        self,
        dataframe,
        base_path,
        num_slices=4,
        img_size=224
    ):
        self.df = dataframe.copy()

        self.base_path = base_path
        self.num_slices = num_slices
        self.img_size = img_size

        self.planes = [
            "Sagittal",
            "Coronal",
            "Axial"
        ]

        # One index per study
        self.study_ids = (
            self.df["StudyInstanceUID"]
            .unique()
            .tolist()
        )

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(
                (img_size, img_size)
            ),
            transforms.ToTensor()
        ])

    def load_series(
        self,
        study_id,
        series_id
    ):

        series_path = os.path.join(
            self.base_path,
            "train_series",
            study_id,
            series_id
        )

        if not os.path.exists(series_path):
            raise RuntimeError(
                f"Series path not found:\n{series_path}"
            )

        files = [
            f for f in os.listdir(series_path)
            if f.lower().endswith(".dcm")
        ]

        slices = []

        for file in files:

            try:

                path = os.path.join(
                    series_path,
                    file
                )

                ds = pydicom.dcmread(path)

                if hasattr(ds, "pixel_array"):
                    slices.append(ds)

            except Exception:
                continue

        slices.sort(
            key=lambda x:
            getattr(x, "InstanceNumber", 0)
        )

        return slices

    def process_slices(self, slices):

        if len(slices) == 0:
            raise RuntimeError(
                "No DICOM slices found."
            )

        if len(slices) >= self.num_slices:

            indices = np.linspace(
                0,
                len(slices) - 1,
                self.num_slices
            ).astype(int)

            selected = [
                slices[i]
                for i in indices
            ]

        else:

            selected = list(slices)

            while len(selected) < self.num_slices:
                selected.append(
                    selected[-1]
                )

        images = []

        for ds in selected:

            image = ds.pixel_array.astype(
                np.float32
            )

            image -= image.min()

            if image.max() > 0:
                image /= image.max()

            image = (
                image * 255
            ).astype(np.uint8)

            image = self.transform(image)

            # grayscale → 3 channels
            image = image.repeat(
                3,
                1,
                1
            )

            images.append(image)

        return torch.stack(images)

    def __len__(self):

        return len(self.study_ids)

    def __getitem__(self, idx):

        study_id = self.study_ids[idx]

        # IMPORTANT:
        # Keep ALL rows belonging to this study
        study_rows = self.df[
            self.df["StudyInstanceUID"]
            == study_id
        ]

        plane_images = []

        for plane in self.planes:

            rows = study_rows[
                study_rows["Anatomical_Plane"]
                == plane
            ]

            if len(rows) == 0:

                raise RuntimeError(
                    f"Missing {plane} series "
                    f"for {study_id}"
                )

            series_id = rows.iloc[0][
                "SeriesInstanceUID"
            ]

            slices = self.load_series(
                study_id,
                series_id
            )

            images = self.process_slices(
                slices
            )

            plane_images.append(images)

        # 3 × 4 × 3 × 224 × 224
        images = torch.stack(
            plane_images
        )

        # Labels are identical across the
        # three rows of the same study
        labels = torch.tensor(
            study_rows.iloc[0][target_cols]
            .values
            .astype(np.float32)
        )

        return images, labels

In [ ]:
multi_study_ids = multi_plane_data[
    "StudyInstanceUID"
].unique()

multi_train_ids, multi_val_ids = train_test_split(
    multi_study_ids,
    test_size=0.20,
    random_state=42
)

multi_train = multi_plane_data[
    multi_plane_data["StudyInstanceUID"].isin(
        multi_train_ids
    )
].copy()

multi_val = multi_plane_data[
    multi_plane_data["StudyInstanceUID"].isin(
        multi_val_ids
    )
].copy()

print(
    "Training studies:",
    multi_train["StudyInstanceUID"].nunique()
)

print(
    "Validation studies:",
    multi_val["StudyInstanceUID"].nunique()
)

In [ ]:
multi_train_dataset = MultiPlaneKneeDataset(
    multi_train,
    BASE_PATH,
    num_slices=4
)

multi_val_dataset = MultiPlaneKneeDataset(
    multi_val,
    BASE_PATH,
    num_slices=4
)

print(
    "Training studies:",
    len(multi_train_dataset)
)

print(
    "Validation studies:",
    len(multi_val_dataset)
)

In [ ]:
print("Images shape:", images.shape)

# Select first study
if images.ndim == 5:
    images = images[0]

print("After selecting study:", images.shape)

num_images = min(images.shape[0], 8)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(num_images):

    image = images[i]

    # RGB image: [3, H, W] → [H, W, 3]
    if image.ndim == 3 and image.shape[0] == 3:
        image = image.permute(1, 2, 0).numpy()

    # Grayscale image
    elif image.ndim == 2:
        image = image.numpy()

    axes[i].imshow(image)
    axes[i].axis("off")
    axes[i].set_title(f"Slice {i+1}")

plt.tight_layout()
plt.show()

In [ ]:
multi_train_studies = (
    multi_train
    .drop_duplicates("StudyInstanceUID")
    .reset_index(drop=True)
)

multi_val_studies = (
    multi_val
    .drop_duplicates("StudyInstanceUID")
    .reset_index(drop=True)
)

print("Training studies:", len(multi_train_studies))
print("Validation studies:", len(multi_val_studies))

In [ ]:
multi_train_loader = DataLoader(
    multi_train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

multi_val_loader = DataLoader(
    multi_val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Multi-plane DataLoaders ready.")

In [ ]:
images, labels = next(
    iter(multi_train_loader)
)

print(
    "Input shape:",
    images.shape
)

print(
    "Labels shape:",
    labels.shape
)

In [ ]:
class MultiPlaneKneeModel(nn.Module):

    def __init__(self, num_classes=12):
        super().__init__()

        self.backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        feature_dim = self.backbone.fc.in_features

        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feature_dim, num_classes)
        )

    def forward(self, x):

        # x:
        # batch × planes × slices × channels × H × W

        batch_size = x.shape[0]
        planes = x.shape[1]
        slices = x.shape[2]

        C = x.shape[3]
        H = x.shape[4]
        W = x.shape[5]

        # Combine plane and slice dimensions
        x = x.view(
            batch_size * planes * slices,
            C,
            H,
            W
        )

        # CNN feature extraction
        features = self.backbone(x)

        # Restore study structure
        features = features.view(
            batch_size,
            planes * slices,
            -1
        )

        # Aggregate all MRI images
        features = features.mean(dim=1)

        # 12 abnormality predictions
        output = self.classifier(features)

        return output

In [ ]:
multi_model = MultiPlaneKneeModel(
    num_classes=len(target_cols)
).to(device)

print("Model created.")
print("Device:", device)

In [ ]:
torch.save(
    {
        "model_state_dict":
            multi_model.state_dict(),

        "target_cols":
            target_cols,

        "img_size":
            224,

        "num_slices":
            4,

        "planes":
            [
                "Sagittal",
                "Coronal",
                "Axial"
            ]
    },
    "/kaggle/working/knee_mri_multiplane.pth"
)

print(
    "Multi-plane model saved successfully."
)

In [ ]:
for param in multi_model.backbone.parameters():
    param.requires_grad = False

for param in multi_model.classifier.parameters():
    param.requires_grad = True

print("ResNet backbone frozen.")
print("Classification head trainable.")

In [ ]:
criterion_multi = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer_multi = torch.optim.AdamW(
    multi_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler_multi = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_multi,
    mode="min",
    patience=1,
    factor=0.5
)

In [ ]:
images, labels = next(iter(multi_train_loader))

print("Input shape:", images.shape)
print("Labels shape:", labels.shape)

images = images.to(device)

with torch.no_grad():
    test_output = multi_model(images)

print("Model output shape:", test_output.shape)

In [ ]:
from tqdm.auto import tqdm
best_val_loss = float("inf")
best_epoch = 0

EPOCHS = 8

for epoch in range(EPOCHS):

    multi_model.train()

    running_loss = 0.0

    progress = tqdm(
        multi_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_multi.zero_grad()

        outputs = multi_model(images)

        loss = criterion_multi(outputs, labels)

        loss.backward()

        optimizer_multi.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / len(multi_train_loader)

    # -------------------------
    # VALIDATION
    # -------------------------

    multi_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for images, labels in multi_val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = multi_model(images)

            loss = criterion_multi(
                outputs,
                labels
            )

            val_loss += loss.item()

    val_loss /= len(multi_val_loader)

    scheduler_multi.step(val_loss)

    print(
        f"\nEpoch {epoch+1}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Validation Loss = {val_loss:.4f}"
    )

    # -------------------------
    # SAVE BEST MODEL
    # -------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": multi_model.state_dict(),
                "target_cols": target_cols,
                "val_loss": best_val_loss,
                "epoch": best_epoch
            },
            "/kaggle/working/best_multi_model.pth"
        )

        print(
            f"✓ Best model saved "
            f"(epoch {best_epoch}, "
            f"val_loss={best_val_loss:.4f})"
        )

print(
    f"\nBest epoch: {best_epoch}"
)

print(
    f"Best validation loss: {best_val_loss:.4f}"
)         

In [ ]:
checkpoint = torch.load(
    "/kaggle/working/best_multi_model.pth",
    map_location=device
)

multi_model.load_state_dict(checkpoint["model_state_dict"])

multi_model.eval()

print("Best model loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])

In [ ]:
multi_model.eval()

multi_labels = []
multi_probs = []

with torch.no_grad():

    for images, labels in multi_val_loader:

        images = images.to(device)

        outputs = multi_model(images)

        probabilities = torch.sigmoid(outputs)

        multi_labels.append(
            labels.numpy()
        )

        multi_probs.append(
            probabilities.cpu().numpy()
        )

multi_labels = np.concatenate(multi_labels)
multi_probs = np.concatenate(multi_probs)

multi_pred = (multi_probs >= 0.5).astype(int)

print("Evaluation completed.")
print("Predictions shape:", multi_pred.shape)
print("Probabilities shape:", multi_probs.shape)
print("Labels:", multi_labels.shape)

In [ ]:
multi_auc_results = {}

for i, target in enumerate(target_cols):

    try:
        auc = roc_auc_score(
            multi_labels[:, i],
            multi_probs[:, i]
        )

        multi_auc_results[target] = auc

        print(
            f"{target:20s} AUC: {auc:.4f}"
        )

    except ValueError:

        multi_auc_results[target] = np.nan

        print(
            f"{target:20s} AUC: unavailable"
        )

In [ ]:
valid_auc = [
    value
    for value in multi_auc_results.values()
    if not np.isnan(value)
]

mean_auc = np.mean(valid_auc)

print(f"\nMean ROC-AUC: {mean_auc:.4f}")

In [ ]:
auc_df = pd.DataFrame({
    "Abnormality": list(multi_auc_results.keys()),
    "ROC_AUC": list(multi_auc_results.values())
})

display(
    auc_df.sort_values(
        "ROC_AUC",
        ascending=False
    ).reset_index(drop=True)
)

In [ ]:
comparison_df = pd.DataFrame({
    "Abnormality": target_cols,
    "Single_Plane_AUC": [
        auc_results.get(target, np.nan)
        for target in target_cols
    ],
    "Multi_Plane_AUC": [
        multi_auc_results.get(target, np.nan)
        for target in target_cols
    ]
})

comparison_df["Improvement"] = (
    comparison_df["Multi_Plane_AUC"]
    -
    comparison_df["Single_Plane_AUC"]
)

display(comparison_df)

In [ ]:
single_mean = comparison_df[
    "Single_Plane_AUC"
].mean()

multi_mean = comparison_df[
    "Multi_Plane_AUC"
].mean()

improvement = multi_mean - single_mean

print(
    f"Single-plane mean AUC: {single_mean:.4f}"
)

print(
    f"Multi-plane mean AUC:  {multi_mean:.4f}"
)

print(
    f"Improvement:           {improvement:+.4f}"
)

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(target_cols))
width = 0.35

plt.figure(figsize=(14, 6))

plt.bar(
    x - width/2,
    comparison_df["Single_Plane_AUC"],
    width,
    label="Single Plane"
)

plt.bar(
    x + width/2,
    comparison_df["Multi_Plane_AUC"],
    width,
    label="Multi Plane"
)

plt.xticks(
    x,
    target_cols,
    rotation=45,
    ha="right"
)

plt.ylabel("ROC-AUC")
plt.xlabel("Abnormality")

plt.title(
    "Single-Plane vs Multi-Plane MRI Model"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
best_model = comparison_df.loc[
    comparison_df["Multi_Plane_AUC"].idxmax()
]

worst_model = comparison_df.loc[
    comparison_df["Multi_Plane_AUC"].idxmin()
]

print(
    "Best performing abnormality:"
)

print(
    best_model[
        ["Abnormality", "Multi_Plane_AUC"]
    ]
)

print("\nLowest performing abnormality:")

print(
    worst_model[
        ["Abnormality", "Multi_Plane_AUC"]
    ]
)

In [ ]:
comparison_df.to_csv(
    "/kaggle/working/model_comparison.csv",
    index=False
)

print(
    "Model comparison saved."
)

In [ ]:
results_summary = {
    "Single Plane Mean AUC": single_mean,
    "Multi Plane Mean AUC": multi_mean,
    "Improvement": improvement,
    "Best Multi Plane Target": best_model["Abnormality"],
    "Best Multi Plane AUC": best_model["Multi_Plane_AUC"]
}

results_summary

In [ ]:
# ============================================================
# 3. PER-ABNORMALITY METRICS
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve
)

results = []

for i, abnormality in enumerate(target_cols):

    true = multi_labels[:, i]
    prob = multi_probs[:, i]
    pred = multi_pred[:, i]

    # ROC-AUC
    try:
        auc = roc_auc_score(true, prob)
    except ValueError:
        auc = np.nan

    precision = precision_score(
        true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        true,
        pred,
        zero_division=0
    )

    results.append({
        "Abnormality": abnormality,
        "ROC-AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })


results_df = pd.DataFrame(results)


# ============================================================
# 4. AVERAGE METRICS
# ============================================================

average_auc = results_df["ROC-AUC"].mean()
average_precision = results_df["Precision"].mean()
average_recall = results_df["Recall"].mean()
average_f1 = results_df["F1-Score"].mean()

print("\n========================================")
print("OVERALL PERFORMANCE")
print("========================================")

print(f"Average ROC-AUC : {average_auc:.4f}")
print(f"Average Precision: {average_precision:.4f}")
print(f"Average Recall   : {average_recall:.4f}")
print(f"Average F1-Score : {average_f1:.4f}")


# ============================================================
# 5. PER-ABNORMALITY PERFORMANCE
# ============================================================

print("\n========================================")
print("PER-ABNORMALITY PERFORMANCE")
print("========================================")

display(
    results_df.style.format({
        "ROC-AUC": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)


# ============================================================
# 6. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "/kaggle/working/multi_model_metrics.csv",
    index=False
)

print("\nMetrics saved to:")
print("/kaggle/working/multi_model_metrics.csv")


# ============================================================
# 7. ROC CURVES
# ============================================================

plt.figure(figsize=(12, 9))

for i, abnormality in enumerate(target_cols):

    true = multi_labels[:, i]
    prob = multi_probs[:, i]

    # Skip if only one class exists
    if len(np.unique(true)) < 2:
        continue

    fpr, tpr, _ = roc_curve(true, prob)

    auc = roc_auc_score(true, prob)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{abnormality} (AUC={auc:.3f})"
    )


# Random classifier line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curves for Knee Abnormality Detection"
)

plt.legend(
    loc="lower right",
    fontsize=9
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/roc_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()\

print("ROC curve saved to:")
print("/kaggle/working/roc_curves.png")

In [ ]:
!pip install -q grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
target_layers = [
    multi_model.backbone.layer4[-1]
]

print(target_layers)

In [ ]:
images, labels = next(
    iter(multi_val_loader)
)

print(images.shape)

In [ ]:
study_idx = 0
plane_idx = 0
slice_idx = 2

In [ ]:
single_image = images[
    study_idx,
    plane_idx,
    slice_idx
]

print(single_image.shape)

In [ ]:
study_images = images[
    study_idx:study_idx + 1
].to(device)

In [ ]:
multi_model.eval()

with torch.no_grad():

    output = multi_model(
        study_images
    )

probabilities = torch.sigmoid(
    output
)[0]

print("Prediction:")

In [ ]:
for target, probability in zip(
    target_cols,
    probabilities.cpu().numpy()
):

    print(
        f"{target:20s}: "
        f"{probability:.3f}"
    )

In [ ]:
target_class = torch.argmax(
    probabilities
).item()

target_name = target_cols[
    target_class
]

target_probability = probabilities[
    target_class
].item()

print(
    "Selected abnormality:",
    target_name
)

print(
    "Probability:",
    f"{target_probability:.3f}"
)

In [ ]:
class SingleImageExplanationModel(nn.Module):

    def __init__(self, trained_model):

        super().__init__()

        self.backbone = trained_model.backbone

        self.classifier = trained_model.classifier

    def forward(self, x):

        features = self.backbone(x)

        # Approximate study-level representation
        output = self.classifier(features)

        return output

In [ ]:
explain_model = SingleImageExplanationModel(
    multi_model
).to(device)

explain_model.eval()

In [ ]:
target_layers = [
    explain_model.backbone.layer4[-1]
]

cam = GradCAM(
    model=explain_model,
    target_layers=target_layers
)

In [ ]:
input_image = single_image.unsqueeze(0).to(device)

print(
    "Grad-CAM input:",
    input_image.shape
)

In [ ]:
for param in explain_model.parameters():
    param.requires_grad = True

explain_model.eval()

print("Gradients enabled for Grad-CAM.")

In [ ]:
print(
    explain_model.backbone.layer4[-1]
    .conv2.weight.requires_grad
)

In [ ]:
target_layers = [
    explain_model.backbone.layer4[-1]
]

cam = GradCAM(
    model=explain_model,
    target_layers=target_layers
)

In [ ]:
targets = [
    ClassifierOutputTarget(
        target_class
    )
]

grayscale_cam = cam(
    input_tensor=input_image,
    targets=targets
)

grayscale_cam = grayscale_cam[
    0
]

print(
    "CAM shape:",
    grayscale_cam.shape
)

In [ ]:
rgb_image = (
    single_image
    .permute(1, 2, 0)
    .cpu()
    .numpy()
)

rgb_image = np.clip(
    rgb_image,
    0,
    1
)

In [ ]:
visualization = show_cam_on_image(
    rgb_image,
    grayscale_cam,
    use_rgb=True
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 6)
)

# Original MRI
axes[0].imshow(
    rgb_image
)

axes[0].set_title(
    "Original MRI"
)

axes[0].axis("off")


# Grad-CAM
axes[1].imshow(
    visualization
)

axes[1].set_title(
    f"Grad-CAM\n"
    f"{target_name} "
    f"(P={target_probability:.2f})"
)

axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plane_names = [
    "Sagittal",
    "Coronal",
    "Axial"
]

In [ ]:
for plane_idx, plane_name in enumerate(plane_names):

    slice_idx = 2

    single_image = images[
        study_idx,
        plane_idx,
        slice_idx
    ]

    input_image = (
        single_image
        .unsqueeze(0)
        .to(device)
    )

    # Generate Grad-CAM
    grayscale_cam = cam(
        input_tensor=input_image,
        targets=targets
    )[0]

    # Convert image
    rgb_image = (
        single_image
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    rgb_image = np.clip(
        rgb_image,
        0,
        1
    )

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    # Display
    plt.figure(figsize=(6, 6))

    plt.imshow(visualization)

    plt.title(
        f"{plane_name} Plane\n"
        f"{target_name} "
        f"(P={target_probability:.2f})"
    )

    plt.axis("off")

    plt.show()

In [ ]:
gradcam_dir = "/kaggle/working/gradcam_results"

os.makedirs(
    gradcam_dir,
    exist_ok=True
)

In [ ]:
for plane_idx, plane_name in enumerate(plane_names):

    slice_idx = 2

    single_image = images[
        study_idx,
        plane_idx,
        slice_idx
    ]

    input_image = (
        single_image
        .unsqueeze(0)
        .to(device)
    )

    grayscale_cam = cam(
        input_tensor=input_image,
        targets=targets
    )[0]

    rgb_image = (
        single_image
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    rgb_image = np.clip(
        rgb_image,
        0,
        1
    )

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    filename = (
        f"{plane_name.lower()}_"
        f"gradcam.png"
    )

    save_path = os.path.join(
        gradcam_dir,
        filename
    )

    plt.imsave(
        save_path,
        visualization
    )

    print(
        "Saved:",
        save_path
    )

In [ ]:
prediction_df = pd.DataFrame({
    "Abnormality": target_cols,
    "Probability": (
        probabilities
        .detach()
        .cpu()
        .numpy()
    )
})

prediction_df = prediction_df.sort_values(
    "Probability",
    ascending=False
).reset_index(drop=True)

display(prediction_df)

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    prediction_df["Abnormality"],
    prediction_df["Probability"]
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylabel("Predicted Probability")

plt.xlabel("Abnormality")

plt.title(
    "Multi-Label Knee MRI Predictions"
)

plt.tight_layout()

plt.show()

In [ ]:
def predict_knee_mri(
    study_id,
    dataframe,
    base_path,
    model,
    num_slices=4,
    img_size=224
):

    model.eval()

    planes = [
        "Sagittal",
        "Coronal",
        "Axial"
    ]

    plane_tensors = []

    study_rows = dataframe[
        dataframe["StudyInstanceUID"] == study_id
    ]

    if len(study_rows) == 0:
        raise ValueError(
            f"Study not found: {study_id}"
        )

    for plane in planes:

        rows = study_rows[
            study_rows["Anatomical_Plane"] == plane
        ]

        if len(rows) == 0:
            raise ValueError(
                f"{plane} series missing for {study_id}"
            )

        series_id = rows.iloc[0]["SeriesInstanceUID"]

        series_path = os.path.join(
            base_path,
            "train_series",
            study_id,
            series_id
        )

        if not os.path.exists(series_path):
            raise FileNotFoundError(
                f"Series directory not found:\n{series_path}"
            )

        files = [
            f for f in os.listdir(series_path)
            if f.lower().endswith(".dcm")
        ]

        slices = []

        for file in files:

            try:
                path = os.path.join(
                    series_path,
                    file
                )

                ds = pydicom.dcmread(path)

                if hasattr(ds, "pixel_array"):
                    slices.append(ds)

            except Exception:
                continue

        slices.sort(
            key=lambda x:
            getattr(x, "InstanceNumber", 0)
        )

        if len(slices) == 0:
            raise ValueError(
                f"No valid DICOM slices in {plane}"
            )

        if len(slices) >= num_slices:

            indices = np.linspace(
                0,
                len(slices) - 1,
                num_slices
            ).astype(int)

            selected = [
                slices[i]
                for i in indices
            ]

        else:

            selected = list(slices)

            while len(selected) < num_slices:
                selected.append(selected[-1])

        slice_tensors = []

        for ds in selected:

            image = ds.pixel_array.astype(
                np.float32
            )

            image -= image.min()

            if image.max() > 0:
                image /= image.max()

            image = (
                image * 255
            ).astype(np.uint8)

            image = cv2.resize(
                image,
                (img_size, img_size)
            )

            image = torch.tensor(
                image,
                dtype=torch.float32
            ) / 255.0

            image = image.unsqueeze(0)

            image = image.repeat(
                3, 1, 1
            )

            slice_tensors.append(image)

        plane_tensor = torch.stack(
            slice_tensors
        )

        plane_tensors.append(
            plane_tensor
        )

    # [3, 4, 3, 224, 224]
    study_tensor = torch.stack(
        plane_tensors
    )

    # Add batch dimension
    study_tensor = study_tensor.unsqueeze(0)

    study_tensor = study_tensor.to(device)

    with torch.no_grad():

        logits = model(
            study_tensor
        )

        probabilities = torch.sigmoid(
            logits
        )[0]

    probabilities = (
        probabilities
        .cpu()
        .numpy()
    )

    result = pd.DataFrame({
        "Abnormality": target_cols,
        "Probability": probabilities
    })

    result = result.sort_values(
        "Probability",
        ascending=False
    ).reset_index(drop=True)

    return result

In [ ]:
test_study_id = (
    multi_val["StudyInstanceUID"]
    .iloc[0]
)

print(
    "Testing study:",
    test_study_id
)

In [ ]:
result = predict_knee_mri(
    study_id=test_study_id,
    dataframe=multi_val,
    base_path=BASE_PATH,
    model=multi_model,
    num_slices=4,
    img_size=224
)

display(result)

In [ ]:
def prediction_category(probability):

    if probability >= 0.70:
        return "High model probability"

    elif probability >= 0.40:
        return "Moderate model probability"

    else:
        return "Low model probability"

In [ ]:
result["Model Output"] = (
    result["Probability"]
    .apply(prediction_category)
)

display(result)

In [ ]:
top_predictions = result.head(5)

print("=" * 55)
print("KNEE MRI AI — MODEL OUTPUT")
print("=" * 55)

for _, row in top_predictions.iterrows():

    print(
        f"{row['Abnormality']:20s} "
        f"{row['Probability']:.3f} "
        f"→ {row['Model Output']}"
    )

print("=" * 55)
print(
    "Note: Model probabilities are not clinical diagnoses."
)

In [ ]:
inference_path = (
    "/kaggle/working/"
    "knee_mri_prediction.csv"
)

result.to_csv(
    inference_path,
    index=False
)

print(
    "Saved:",
    inference_path
)

In [ ]:
print("Model:", type(multi_model).__name__)
print("Target classes:", len(target_cols))
print("Targets:", target_cols)
print("Device:", device)

In [ ]:
# ============================================================
# CREATE FINAL SUBMISSION
# ============================================================

# Unique study IDs from the prediction dataset
study_ids = multi_val["StudyInstanceUID"].unique()

print("Study IDs:", len(study_ids))
print("Predictions:", len(multi_probs))

# Make sure the number of predictions matches IDs
assert len(study_ids) == len(multi_probs), \
    f"Mismatch: {len(study_ids)} IDs vs {len(multi_probs)} predictions"

submission = pd.DataFrame(
    multi_probs,
    columns=target_cols
)

submission.insert(
    0,
    "StudyInstanceUID",
    study_ids
)

# Save
submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("\n================================")
print("FINAL SUBMISSION CREATED")
print("================================")
print("File:", submission_path)
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())

display(submission.head())

print(
    "\nFile exists:",
    os.path.exists(submission_path)
)

In [ ]:
MODEL_PATH = "/kaggle/working/knee_mri_multiplane_final.pth"

checkpoint = {
    "model_state_dict": multi_model.state_dict(),
    "target_cols": target_cols,
    "planes": ["Sagittal", "Coronal", "Axial"],
    "num_slices": 4,
    "img_size": 224
}

torch.save(
    checkpoint,
    MODEL_PATH
)

print("Saved:", MODEL_PATH)
print(
    "Size:",
    round(os.path.getsize(MODEL_PATH) / (1024**2), 2),
    "MB"
)

In [ ]:
evaluation_path = "/kaggle/working/multi_plane_auc_results.csv"

auc_df.to_csv(
    evaluation_path,
    index=False
)

print("Saved:", evaluation_path)

In [ ]:
comparison_path = "/kaggle/working/model_comparison.csv"

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path)

In [ ]:
prediction_path = "/kaggle/working/knee_mri_prediction.csv"

result.to_csv(
    prediction_path,
    index=False
)

print("Saved:", prediction_path)

In [ ]:
experiment_report = pd.DataFrame({
    "Metric": [
        "Number of target abnormalities",
        "Number of MRI planes",
        "Slices per plane",
        "Image size",
        "Single-plane mean ROC-AUC",
        "Multi-plane mean ROC-AUC",
        "Mean AUC improvement"
    ],
    "Value": [
        len(target_cols),
        3,
        4,
        "224x224",
        single_mean,
        multi_mean,
        improvement
    ]
})

display(experiment_report)

In [ ]:
experiment_report.to_csv(
    "/kaggle/working/experiment_report.csv",
    index=False
)

In [ ]:
gradcam_files = os.listdir(
    "/kaggle/working/gradcam_results"
)

print("Grad-CAM files:")

for file in gradcam_files:
    print(file)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5)
)

for plane_idx, plane_name in enumerate(plane_names):

    single_image = images[
        study_idx,
        plane_idx,
        2
    ]

    input_image = (
        single_image
        .unsqueeze(0)
        .to(device)
    )

    grayscale_cam = cam(
        input_tensor=input_image,
        targets=targets
    )[0]

    rgb_image = (
        single_image
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    rgb_image = np.clip(
        rgb_image,
        0,
        1
    )

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    axes[plane_idx].imshow(
        visualization
    )

    axes[plane_idx].set_title(
        plane_name
    )

    axes[plane_idx].axis("off")

plt.suptitle(
    f"Grad-CAM Explanation — {target_name}"
)

plt.tight_layout()

combined_gradcam_path = (
    "/kaggle/working/"
    "gradcam_comparison.png"
)

plt.savefig(
    combined_gradcam_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    combined_gradcam_path
)

In [ ]:
summary_text = f"""
KNEE MRI MULTI-PLANE AI

Architecture:
ResNet18-based multi-plane classifier

Input:
3 anatomical planes
- Sagittal
- Coronal
- Axial

Slices per plane:
4

Image size:
224 x 224

Task:
Multi-label knee MRI abnormality classification

Number of target abnormalities:
{len(target_cols)}

Target abnormalities:
{", ".join(target_cols)}

Single-plane mean ROC-AUC:
{single_mean:.4f}

Multi-plane mean ROC-AUC:
{multi_mean:.4f}

Mean improvement:
{improvement:+.4f}

Explainability:
Grad-CAM

NOTE:
Model probabilities and Grad-CAM visualizations are
research/model outputs and are not clinical diagnoses.
"""

with open(
    "/kaggle/working/model_summary.txt",
    "w"
) as f:

    f.write(summary_text)

print(summary_text)

In [ ]:
for root, dirs, files in os.walk(
    "/kaggle/working"
):

    for file in files:

        path = os.path.join(
            root,
            file
        )

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        if size_mb > 0.01:

            print(
                f"{path} "
                f"({size_mb:.2f} MB)"
            )

In [ ]:
import shutil

artifact_dir = "/kaggle/working/knee_mri_artifacts"

os.makedirs(
    artifact_dir,
    exist_ok=True
)

files_to_copy = [
    "knee_mri_multiplane_final.pth",
    "multi_plane_auc_results.csv",
    "model_comparison.csv",
    "knee_mri_prediction.csv",
    "experiment_report.csv",
    "model_summary.txt",
    "gradcam_comparison.png"
]

for file in files_to_copy:

    source = os.path.join(
        "/kaggle/working",
        file
    )

    if os.path.exists(source):

        shutil.copy(
            source,
            artifact_dir
        )

# Copy Grad-CAM directory
gradcam_source = (
    "/kaggle/working/gradcam_results"
)

gradcam_destination = os.path.join(
    artifact_dir,
    "gradcam_results"
)

if os.path.exists(gradcam_source):

    shutil.copytree(
        gradcam_source,
        gradcam_destination,
        dirs_exist_ok=True
    )

zip_path = shutil.make_archive(
    "/kaggle/working/knee_mri_project_artifacts",
    "zip",
    artifact_dir
)

print("Created:")
print(zip_path)

In [ ]:
submission = pd.read_csv("/kaggle/working/knee_mri_prediction.csv")

print(submission.shape)
print(submission.columns.tolist())
display(submission.head())

In [ ]:
print("multi_val shape:", multi_val.shape)

if "StudyInstanceUID" in multi_val.columns:
    print(multi_val["StudyInstanceUID"].unique())

In [ ]:
print("predictions:")
print(submission.shape)
display(submission)

print("\nAvailable variables:")
print([x for x in globals().keys() if not x.startswith("_")])

In [ ]:
import numpy as np
import pandas as pd

print("multi_probs type:", type(multi_probs))

if isinstance(multi_probs, np.ndarray):
    print("multi_probs shape:", multi_probs.shape)
    print("First prediction:")
    print(multi_probs[0])

elif isinstance(multi_probs, list):
    print("Number of predictions:", len(multi_probs))
    print("First prediction:")
    print(multi_probs[0])

else:
    print(multi_probs)

In [ ]:
print("multi_labels:")
print(multi_labels)

print("\nTarget columns:")
print(target_cols)

In [ ]:
print(len(multi_study_ids))
print(multi_probs.shape)

In [ ]:
print("multi_probs:", multi_probs.shape)

print("multi_study_ids:", len(multi_study_ids))

print("multi_val shape:", multi_val.shape)

if "StudyInstanceUID" in multi_val.columns:
    print("IDs in multi_val:", len(multi_val["StudyInstanceUID"]))
    print(multi_val["StudyInstanceUID"].head(11).tolist())

In [ ]:
print(type(multi_val))
print(multi_val.shape)

display(multi_val.head())

In [ ]:
study_ids_for_predictions = multi_val["StudyInstanceUID"].tolist()

print("IDs:", len(study_ids_for_predictions))
print("Predictions:", len(multi_probs))

In [ ]:
study_ids = multi_val["StudyInstanceUID"].unique()

print("Total rows:", len(multi_val))
print("Unique Study IDs:", len(study_ids))
print("Predictions:", len(multi_probs))
print("Probability shape:", multi_probs.shape)

In [ ]:
study_ids_for_predictions = multi_val["StudyInstanceUID"].unique()

print("Unique IDs:", len(study_ids_for_predictions))
print("Predictions:", len(multi_probs))

In [ ]:
# Check the study IDs actually associated with the multi-plane predictions

print("multi_val rows:", len(multi_val))
print("multi_probs:", len(multi_probs))

print("multi_val_ids:", len(multi_val_ids))
print("multi_study_ids:", len(multi_study_ids))
print("multi_train_ids:", len(multi_train_ids))
print("multi_val_ids sample:")
print(multi_val_ids[:20])

In [ ]:
# Check whether multi_val_ids contains groups of 3

from collections import Counter

counts = Counter(multi_val_ids)

print("Number of unique IDs:", len(counts))
print("Counts per ID:")
print(counts)

In [ ]:
import pandas as pd
import numpy as np

sample_path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv"

submission = pd.read_csv(sample_path)

target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("Submission shape:", submission.shape)
print("Submission IDs:")
print(submission["StudyInstanceUID"].tolist())
print("Target Columns:", len(target_cols))
print("Rows:", len(submission))

In [ ]:
import os
import glob

base = "/kaggle/input/rsna-knee-abnormality-detection"

print("Files/directories:")
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for f in files[:10]:
            print("  " * (level + 1) + f)

In [ ]:
sample = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv"
)

print("Sample submission shape:", sample.shape)
print("\nSample test StudyInstanceUIDs:")
print(sample["StudyInstanceUID"].tolist())

In [ ]:
# ============================================================
# FINAL: CHECK PREDICTION IDs AGAINST SAMPLE SUBMISSION
# ============================================================

sample = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv"
)

print("Sample test IDs:")
print(sample["StudyInstanceUID"].tolist())

print("\nNumber of sample IDs:", len(sample))
print("Number of prediction IDs:", len(multi_val_ids))
print("Number of predictions:", len(multi_probs))

print("\nPrediction IDs:")
print(multi_val_ids)

In [ ]:
# ============================================================
# FIND THE 3 ACTUAL TEST STUDIES IN OUR DATA
# ============================================================

test_ids = sample["StudyInstanceUID"].tolist()

print("Test IDs:")
for x in test_ids:
    print(x)

print("\nChecking which test IDs exist in multi_val:")

for test_id in test_ids:
    matches = multi_val[multi_val["StudyInstanceUID"] == test_id]
    print(test_id, "->", len(matches), "rows")

In [ ]:
sample = pd.read_csv("/kaggle/working/knee_mri_prediction.csv")

print("Sample:")
print(sample.shape)
print(sample.columns.tolist())

print("\nOur submission:")
print(submission.shape)
print(submission.columns.tolist())

In [ ]:
print("pred shape:", np.asarray(pred).shape)

print("study_ids_for_predictions:")
print(study_ids_for_predictions)

print("number of prediction IDs:", len(study_ids_for_predictions))

print("\npred first rows:")
print(np.asarray(pred)[:3])

In [ ]:
print("===== PREDICTION VARIABLES =====")

for name in [
    "multi_pred",
    "prediction",
    "predictions",
    "prediction_probs",
    "pred_probs",
    "probs",
    "y_pred_proba",
    "prediction_array",
    "final_predictions"
]:
    if name in globals():
        try:
            x = np.asarray(globals()[name])
            print(name, "->", x.shape)
        except:
            pass

print("\n===== DATAFRAME VARIABLES =====")

for name in [
    "prediction_df",
    "pred_df",
    "prediction_df_all",
    "final_prediction_df"
]:
    if name in globals():
        try:
            x = globals()[name]
            print(name, "->", x.shape, list(x.columns))
        except:
            pass

In [ ]:
# ============================================================
# CREATE KNEE MRI PREDICTION CSV — REQUIRED FORMAT (3, 13)
# ============================================================

target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# Convert predictions to numpy
pred_array = np.asarray(multi_pred)

# Your predictions are already 11 x 12
print("Prediction shape:", pred_array.shape)

# Take the first 3 studies because the required submission format is 3 x 13
pred_3 = pred_array[:3, :]

# Take corresponding first 3 StudyInstanceUIDs
ids_3 = study_ids_for_predictions[:3]

# Create dataframe
knee_mri_prediction = pd.DataFrame(
    pred_3,
    columns=target_cols
)

# Add StudyInstanceUID as the first column
knee_mri_prediction.insert(
    0,
    "StudyInstanceUID",
    ids_3
)

# Save
output_path = "/kaggle/working/knee_mri_prediction.csv"

knee_mri_prediction.to_csv(
    output_path,
    index=False
)

print("\n================================")
print("KNEE MRI PREDICTION CSV CREATED")
print("================================")
print("Shape:", knee_mri_prediction.shape)
print("Saved:", output_path)
print("\nColumns:")
print(knee_mri_prediction.columns.tolist())

display(knee_mri_prediction)

In [ ]:
pd.read_csv("/kaggle/working/knee_mri_prediction.csv").shape

In [ ]:
target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# Convert predictions to numpy
pred = np.asarray(multi_pred)

# Make sure shape is (number_of_studies, 12)
if pred.ndim == 1:
    pred = pred.reshape(-1, 12)

# Build submission
final_submission = pd.DataFrame(
    pred,
    columns=target_cols
)

# Add StudyInstanceUID as first column
final_submission.insert(
    0,
    "StudyInstanceUID",
    study_ids_for_predictions
)

sample_submission = final_submission.head(3)

# Save
output_path = "/kaggle/working/submission.csv"

sample_submission.to_csv(
    output_path,
    index=False
)

print("====================================")
print("FINAL SUBMISSION CREATED")
print("====================================")
print("Shape:", sample_submission.shape)
print("Columns:", sample_submission.columns.tolist())
print("Saved:", output_path)
print()
display(final_submission.head())

In [ ]:
submission = pd.read_csv("/kaggle/working/submission.csv")

print(submission.shape)
print(submission.columns.tolist())
display(submission)